## SPARK STREAMING

In [0]:
entities = ['customers', 'drivers', 'vehicles', 'payments', 'locations', 'trips']

In [0]:
for entity in entities:
    df = spark.read.format("csv")\
                        .option("header", True)\
                        .option("inferSchema", True)\
                        .load(f"/Volumes/pysparkdbt/source_data/data_files/{entity}/")
    
    schema_df = df.schema

    df_stream = spark.readStream.format("csv")\
                    .option("header", True)\
                    .schema(schema_df)\
                    .load(f"/Volumes/pysparkdbt/source_data/data_files/{entity}/")

    df_stream.writeStream.format("delta")\
                        .outputMode("append")\
                        .option("checkpointLocation", f"/Volumes/pysparkdbt/bronze/checkpoint/{entity}")\
                        .trigger(once=True)\
                        .toTable(f"pysparkdbt.bronze.{entity}")